In [1]:
# DESCARGA AUTOMÁTICA ERA5 - PRECIPITACIÓN

# INSTALACIÓN DE LIBRERÍA
# ---------------------------------------------------------
# Ejecutar una sola vez en terminal:
#
# pip install cdsapi
#

# ---------------------------------------------------------
# LIBRERÍAS

import cdsapi
import os

# ---------------------------------------------------------
# CREAR DIRECTORIO DE SALIDA

os.makedirs("./data_heavy", exist_ok=True)

# ---------------------------------------------------------
# CONFIGURACIÓN DE CREDENCIALES CDS
credenciales = """
url: https://cds.climate.copernicus.eu/api
key: 6199561d-6712-45f5-9d32-2aa7fc7b8239
"""

# Crear automáticamente el archivo .cdsapirc
with open("/root/.cdsapirc", "w") as archivo:
    archivo.write(credenciales)

print(".cdsapirc creado correctamente")

# ---------------------------------------------------------
# CONFIGURAR CLIENTE CDS
client = cdsapi.Client()

# ---------------------------------------------------------
# DATASET ERA5
dataset = "reanalysis-era5-single-levels"

# ---------------------------------------------------------
# SOLICITUD DE DATOS

request = {

    "product_type": ["reanalysis"],

    "variable": ["total_precipitation"],

    "year": ["2005"],

    "month": ["07"],

    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07"
    ],

    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],

    "data_format": "netcdf",

    "download_format": "unarchived",

    # Norte, Oeste, Sur, Este
    "area": [
        5.25,
        -74.25,
        5.00,
        -74.00
    ]
}

# ---------------------------------------------------------
# ARCHIVO DE SALIDA
archivo_salida = "./data_heavy/era5_precipitacion_pacho.nc"

# ---------------------------------------------------------
# DESCARGA AUTOMÁTICA

print("Iniciando descarga ERA5...")

client.retrieve(
    dataset,
    request,
    archivo_salida
)

# ---------------------------------------------------------
# MENSAJE FINAL

print("===================================")
print("DESCARGA COMPLETADA")
print("===================================")
print(f"Archivo guardado en: {archivo_salida}")

.cdsapirc creado correctamente


2026-05-18 02:40:35,274 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).


Iniciando descarga ERA5...


2026-05-18 02:40:36,552 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-18 02:40:36,554 INFO Request ID is b890fd59-2571-4d2a-89ea-8f99aea6b2a7
2026-05-18 02:40:36,781 INFO status has been updated to accepted
2026-05-18 02:41:00,664 INFO status has been updated to successful


37f5560d6e895a851e772fc3f05bc655.nc:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

DESCARGA COMPLETADA
Archivo guardado en: ./data_heavy/era5_precipitacion_pacho.nc


In [10]:
# LIBRERÍAS

import xarray as xr
import dask
import numpy as np

# ---------------------------------------------------------
# RUTA DEL ARCHIVO

ruta_nc = "./data_heavy/era5_precipitacion_pacho.nc"

# ---------------------------------------------------------
# LECTURA DEL NETCDF (LAZY EVALUATION)

ds = xr.open_dataset(
    ruta_nc,
    engine="h5netcdf",
    chunks={}
)

print("===================================")
print("DATASET CARGADO")
print("===================================")

# ---------------------------------------------------------
# AUDITORÍA DEL HIPERCUBO

print("\nDIMENSIONES:")
print(ds.dims)

print("\nVARIABLES:")
print(ds.data_vars)

print("\nCRS:")
print(ds.attrs)

print("\nRESUMEN GENERAL:")
print(ds)

# ---------------------------------------------------------
# IDENTIFICAR VARIABLE PRINCIPAL

precip = ds["tp"]

# ---------------------------------------------------------
# ÁLGEBRA DE MAPAS (VECTORIALIZADA)

precip_mm = precip * 1000

# Actualizar atributos
precip_mm.attrs["units"] = "mm"

precip_mm.attrs["long_name"] = (
    "Precipitación total"
)

print("\nConversión realizada: m -> mm")

# ---------------------------------------------------------
# REMUESTREO TEMPORAL

precip_diaria = precip_mm.resample(
    valid_time="1D"
).sum()

print("\nRemuestreo temporal completado")

# ---------------------------------------------------------
# MÁXIMO MENSUAL POR PÍXEL

max_mensual = precip_diaria.resample(
    valid_time="1ME"
).max()

max_mensual.name = (
    "precipitacion_maxima_mensual"
)

print("\nMáximo mensual calculado")

# ---------------------------------------------------------
# FECHA DE OCURRENCIA DEL MÁXIMO

# idxmax devuelve la fecha exacta donde ocurrió
# el máximo para cada píxel

fecha_max = precip_diaria.resample(
    valid_time="1ME"
).map(
    lambda x: x.idxmax(dim="valid_time")
)

fecha_max.name = "fecha_maxima_precipitacion"

# ---------------------------------------------------------
# LIMPIAR ATRIBUTOS CONFLICTIVOS - Las fechas NO deben heredar unidades de precipitación

fecha_max.attrs = {
    "long_name": "Fecha de ocurrencia de la precipitación máxima mensual"
}

print("\nFecha de ocurrencia calculada")

# ---------------------------------------------------------
# CREAR DATASET FINAL

ds_final = xr.Dataset({

    "precipitacion_maxima_mensual": max_mensual,

    "fecha_maxima_precipitacion": fecha_max

})

# ---------------------------------------------------------
# EXPORTAR NETCDF FINAL

ruta_salida = (
    "./data_heavy/mi_zona_procesada.nc"
)

ds_final.to_netcdf(
    ruta_salida
)

print("\n===================================")
print("PROCESAMIENTO COMPLETADO")
print("===================================")
print(f"Archivo exportado en:")
print(ruta_salida)



DATASET CARGADO

DIMENSIONES:
FrozenMappingWarningOnValuesAccess({'valid_time': 168, 'latitude': 2, 'longitude': 2})

VARIABLES:
Data variables:
    tp       (valid_time, latitude, longitude) float32 3kB dask.array<chunksize=(168, 2, 2), meta=np.ndarray>

CRS:
{'GRIB_centre': 'ecmf', 'GRIB_centreDescription': 'European Centre for Medium-Range Weather Forecasts', 'GRIB_subCentre': np.int64(0), 'Conventions': 'CF-1.7', 'institution': 'European Centre for Medium-Range Weather Forecasts', 'history': '2026-05-17T03:45 GRIB to CDM+CF via cfgrib-0.9.15.1/ecCodes-2.42.0 with {"source": "tmp2xg3efn7/data.grib", "filter_by_keys": {"stream": ["oper"], "stepType": ["accum"]}, "encode_cf": ["parameter", "time", "geography", "vertical"]}'}

RESUMEN GENERAL:
<xarray.Dataset> Size: 7kB
Dimensions:     (valid_time: 168, latitude: 2, longitude: 2)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1kB 2005-07-01 ... 2005-07-07T23:...
  * latitude    (latitude) float64 16B 5.25 5.0
  * longitude   